# Visual Learning: training on Colab

Trains the MLP/CNN models and stores hidden-layer activations of a fixed test subset (see `README.md`).

1. `Runtime > Change runtime type > T4 GPU`.
2. Set the configurations for this account below, then `Runtime > Run all`.

The code is cloned from GitHub to the Colab disk. Outputs are written to Google Drive, so they survive disconnects.
If the runtime disconnects, run all cells again: finished configurations are skipped and interrupted ones resume from their last epoch.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Get the latest code (clone on first run, pull afterwards).
import os

REPO = 'https://github.com/Andrea-Coa/visual-learning.git'
CODE = '/content/visual-learning'

if os.path.isdir(CODE):
    !git -C {CODE} pull --ff-only
else:
    !git clone --depth 1 {REPO} {CODE}
%cd {CODE}
!git log -1 --oneline

In [ ]:
# Outputs go to Drive; datasets go to the fast local disk.
from google.colab import drive
drive.mount('/content/drive')

OUT = '/content/drive/MyDrive/visual-learning/raw'
DATA = '/content/data'

In [ ]:
# Configurations to train in THIS account (space-separated).
# One account: leave everything. Several accounts: split by dataset, e.g. 'mnist' / 'svhn' / 'cifar10'.
DATASETS = 'mnist svhn cifar10'
MODELS = 'mlp cnn'

In [ ]:
# Optional smoke test (~1 min): 1 short epoch, written to a throw-away folder.
!python -m pipeline.train --dataset mnist --model mlp cnn --epochs 1 --max-batches 200 --data-dir {DATA} --out /content/smoke

In [ ]:
# Full training. Each epoch line prints its duration and the ETA for the current configuration.
!python -m pipeline.train --dataset {DATASETS} --model {MODELS} --data-dir {DATA} --out {OUT}

In [ ]:
# Summary of finished configurations.
import json, pathlib
for path in sorted(pathlib.Path(OUT).glob('*/*/metrics.json')):
    m = json.loads(path.read_text())
    print(f"{m['dataset']:8s} {m['model']}  test acc {m['test']['accuracy']:.4f}  "
          f"(paper {m['paper_test_accuracy']:.4f})  train acc {m['train']['accuracy']:.4f}  "
          f"time {m['training_time_s'] / 60:.1f} min")